In [16]:
%pip install scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [18]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Previous campaign features
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    # Call quality features
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    # Financial features
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['debt'] = (df['balance'] < 0).astype(int)
    # Campaign pressure features
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    # Age groups
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    # Interaction features
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA = add_features(TEST_DATA)

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'long_call',
            'never_contacted', 'very_long_call', 'short_call',
            'has_balance', 'high_balance', 'debt',
            'over_contacted', 'first_contact',
            'is_young', 'is_retired_age', 'is_middle_age',
            'long_call_prev_success', 'long_call_never_contacted',
            'high_balance_long_call']

def preprocess(df, encoder, scaler, poly):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=encoder.get_feature_names_out(),
        index=df.index
    )
    num_scaled = scaler.transform(df[num_cols])
    num_poly = poly.transform(num_scaled)
    num_poly_df = pd.DataFrame(
        num_poly,
        columns=poly.get_feature_names_out(num_cols),
        index=df.index
    )
    return pd.concat([cat_enc, num_poly_df], axis=1)

ENCODER = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(TRAIN_DATA[cat_cols])
SCALER = StandardScaler().fit(TRAIN_DATA[num_cols])
POLY = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False).fit(
    SCALER.transform(TRAIN_DATA[num_cols])
)

X_train = preprocess(TRAIN_DATA, ENCODER, SCALER, POLY)
X_test = preprocess(TEST_DATA, ENCODER, SCALER, POLY)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 382)
X_test shape: (19893, 382)


In [19]:
model = LogisticRegression(
    max_iter=2000, class_weight='balanced',
    C=0.05, penalty='l2', solver='lbfgs', random_state=42
)
cv_scores = cross_val_score(model, X_train, y_train, cv=rskf, scoring='balanced_accuracy', n_jobs=-1)
print(f'CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.

CV Balanced Accuracy: 0.8248 ± 0.0071


In [20]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
I_train, I_val = next(splitter.split(X_train, y_train))
X_tr, X_val = X_train.iloc[I_train], X_train.iloc[I_val]
y_tr, y_val = y_train[I_train], y_train[I_val]

model.fit(X_tr, y_tr)
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold, best_ba = 0.5, 0.0
for thresh in np.arange(0.1, 0.9, 0.01):
    preds = (val_probs >= thresh).astype(int)
    ba = balanced_accuracy_score(y_val, preds)
    if ba > best_ba:
        best_ba = ba
        best_threshold = thresh

print(f'Best threshold: {best_threshold:.2f}')
print(f'Best Val Balanced Accuracy: {best_ba:.4f}')

/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best threshold: 0.50
Best Val Balanced Accuracy: 0.8382


In [21]:
final_model = LogisticRegression(
    max_iter=2000, class_weight='balanced',
    C=0.05, penalty='l2', solver='lbfgs', random_state=42
)
final_model.fit(X_train, y_train)
test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)
print(f'Prediction distribution — 0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')

/Users/dayana/Desktop/3.1학기/ 기계학습/lab/lab1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Prediction distribution — 0: 14916, 1: 4977


save to csv


In [22]:
submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'subscription': test_preds
})

submission.to_csv('submission.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             0
